# Metabolic cost analysis

Reproduces the energy-expenditure (EE) and heart-rate (HR) figures of the paper (pooled and normalised to standing).

Input: COSMED Q-NRG Max Excel exports, one per session (`ses_1..3.xlsx`) plus the 30 min supine resting test (`ree.xlsx`).
The first 9 columns (subject header) are dropped, and each row is a 30 s average.
For every phase, the last `N = 2` rows (the last minute) are taken as steady state.

The participant data used in the paper are **not** included in this repository. See `docs/09_experimental_protocol.md`.

In [ ]:
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

# Folder with ses_1.xlsx, ses_2.xlsx, ses_3.xlsx and ree.xlsx (not included in the repo)
DATA_DIR = Path('data')


In [ ]:
def read_cosmed(path):
    df = pd.read_excel(path)
    df = df.iloc[2:, 9:]          # drop subject header and unit rows
    df['t'] = np.arange(len(df)) * 30 / 60   # 30 s samples -> minutes
    return df


# COSMED Q-NRG exports (30 s averages). Each session file needs a 'Fase' column
# labelling the phases: parado (standing), bastones (crutches), pasiva, activa.
sesiones = [read_cosmed(DATA_DIR / f'ses_{s}.xlsx') for s in [1, 2, 3]]

fases = {'parado': 'Standing', 'bastones': 'Crutches',
         'pasiva': 'Passive', 'activa': 'Active'}

# basal: 30 min supine REE test, steady-state window t = 2-28 min
df_basal = read_cosmed(DATA_DIR / 'ree.xlsx')
basal = {'Basal': df_basal[(df_basal['t'] >= 2) & (df_basal['t'] <= 28)]}

N = 2   # last minute = 2 samples of 30 s

# pooled over the 3 sessions
datos = dict(basal)
datos.update({en: pd.concat([df[df['Fase'] == es].tail(N) for df in sesiones])
              for es, en in fases.items()})

# one dict per session
datos_ses = [dict(basal, **{en: df[df['Fase'] == es].tail(N)
                            for es, en in fases.items()}) for df in sesiones]
# order in which the phases appear in each session
for i, df in enumerate(sesiones, start=1):
    orden = df.loc[df['Fase'].isin(fases), 'Fase'].drop_duplicates()
    print(f'Session {i}: ' + ' -> '.join(fases[f] for f in orden))


In [ ]:
FIGDIR = Path('figuras')
FIGDIR.mkdir(exist_ok=True)


def guardar(fig, nombre):
    """Vector PDF for the manuscript, 600 dpi PNG for previews."""
    for ext in ('pdf', 'png'):
        fig.savefig(FIGDIR / f'{nombre}.{ext}', dpi=600, bbox_inches='tight')


plt.rcParams.update({'font.family': 'serif',
                     'mathtext.fontset': 'dejavuserif',
                     'font.size': 9,
                     'axes.linewidth': 0.8})


def figura(datos, archivo, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.0), dpi=150)

    for ax, letra, col, ylabel in zip(axes, ['A', 'B'],
                                      ['EEkc', 'HR'],
                                      ['Energy expenditure (kcal/day)',
                                       'Heart rate (bpm)']):
        for i, (nombre, d) in enumerate(datos.items()):
            v = d[col].dropna().values
            if len(v) == 0:       # no valid samples (e.g. HR missing)
                continue
            ax.boxplot(v, positions=[i], widths=0.6, tick_labels=[nombre],
                       medianprops=dict(color='black', linewidth=1.2),
                       flierprops=dict(marker='o', markersize=2.5,
                                       markerfacecolor='none', markeredgewidth=0.6))
            ax.annotate(f'{np.mean(v):.0f}', xy=(i, np.max(v)), xytext=(0, 6),
                        textcoords='offset points', ha='center', fontsize=7.5)

        ax.set_ylabel(ylabel)
        ax.set_xlim(-0.6, len(datos) - 0.4)
        ax.margins(y=0.14)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=8)
        ax.set_title(letra, loc='left', fontweight='bold', fontsize=10, pad=6)

    fig.suptitle(titulo, fontsize=10, y=1.02)
    plt.tight_layout()
    guardar(fig, archivo)
    plt.show()




figura(datos, 'gasto_energetico', 'Sessions 1-3 pooled')

order = {
    1: ['Standing', 'Crutches', 'Passive', 'Active'],
    2: ['Standing', 'Passive', 'Active', 'Crutches'],
    3: ['Standing', 'Active', 'Crutches', 'Passive']
}

for i, d in enumerate(datos_ses, start=1):
    print(f'Session {i}: ' + ' -> '.join(order[i]))
    figura(d, f'gasto_energetico_ses{i}', f'Session {i}')

In [ ]:
# Normalized to the Standing level of the *same* session, so that between-session
# shifts (Session 3 runs ~15 bpm lower overall) cancel out.
# Reference = mean over the whole Standing phase of that session, not its last
# minute: Session 2 has no valid HR in the last minute of Standing.

fases_norm = ['Crutches', 'Passive', 'Active']
norm = {col: {f: [] for f in fases_norm} for col in ['EEkc', 'HR']}

for df, d in zip(sesiones, datos_ses):
    for col in ['EEkc', 'HR']:
        ref = pd.to_numeric(df.loc[df['Fase'] == 'parado', col], errors='coerce').mean()
        for f in fases_norm:
            v = pd.to_numeric(d[f][col], errors='coerce').dropna().values
            norm[col][f].append(v / ref)

norm = {col: {f: np.concatenate(v) for f, v in dd.items()} for col, dd in norm.items()}


fig, axes = plt.subplots(1, 2, figsize=(6.0, 3.0), dpi=150)

for ax, letra, col, ylabel in zip(axes, ['A', 'B'],
                                  ['EEkc', 'HR'],
                                  ['Energy expenditure (\u00d7 standing)',
                                   'Heart rate (\u00d7 standing)']):
    ax.axhline(1, color='grey', linestyle='--', linewidth=0.8, zorder=0)
    for i, f in enumerate(fases_norm):
        v = norm[col][f]
        ax.boxplot(v, positions=[i], widths=0.6, tick_labels=[f],
                   medianprops=dict(color='black', linewidth=1.2),
                   flierprops=dict(marker='o', markersize=2.5,
                                   markerfacecolor='none', markeredgewidth=0.6))
        ax.annotate(f'{np.mean(v):.2f}', xy=(i, np.max(v)), xytext=(0, 6),
                    textcoords='offset points', ha='center', fontsize=7.5)

    ax.set_ylabel(ylabel)
    ax.set_xlim(-0.6, len(fases_norm) - 0.4)
    ax.margins(y=0.16)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=8)
    ax.set_title(letra, loc='left', fontweight='bold', fontsize=10, pad=6)

fig.suptitle('Normalized to standing (sessions 1-3 pooled)', fontsize=10, y=1.02)
plt.tight_layout()
guardar(fig, 'normalizada')
plt.show()

for col in ['EEkc', 'HR']:
    print(col, {f: round(float(np.mean(v)), 2) for f, v in norm[col].items()})